# Notebook 04: Feature Engineering, Leakage Guards & Feature Selection
**Project:** Exploratory Data Analysis of Air Pollution in Indian Cities
**Course:** IT 462 — Exploratory Data Analysis | Autumn Semester
**Mentor:** Dr. Gopinath Panda

This notebook implements:
1. Autoregressive multi-day lags and rolling statistics.
2. Leakage guardrails ensuring all predictors strictly use information up to day t.
3. Multi-source feature extraction (Ventilation index, Stubble plume lags, Cyclical calendar).
4. Feature selection using Mutual Information, Lasso regularization, and VIF multicollinearity screening.

In [1]:
import os, sys
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(".").resolve().parents[0] if Path(".").resolve().name == "notebooks" else Path(".").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import TABLES_DIR, PROCESSED_DATA_DIR
print("Project Root:", PROJECT_ROOT)

Project Root: /Users/Tanvi/Desktop/air_quality_eda


## 1. Feature Dictionary & Engineering Formulations

In [2]:
df_dict = pd.read_csv(TABLES_DIR / "feature_dictionary.csv")
df_dict

,Feature,Formula,Domain,Leakage_Safe
0,pm25_lag_1,PM2.5(t-1),Autoregressive,Yes
1,pm25_lag_2,PM2.5(t-2),Autoregressive,Yes
2,pm25_lag_3,PM2.5(t-3),Autoregressive,Yes
3,pm25_lag_7,PM2.5(t-7),Weekly Seasonality,Yes
4,pm25_roll3_mean,Mean(PM2.5[t-3:t-1]),Short-term Baseline,Yes
5,pm25_roll7_mean,Mean(PM2.5[t-7:t-1]),Weekly Baseline,Yes
6,pm25_roll30_mean,Mean(PM2.5[t-30:t-1]),Monthly Baseline,Yes
7,pm25_diff_1,PM2.5(t) - PM2.5(t-1),Momentum / Delta,Yes
8,ventilation_index,(WS / 3.6) * BLH,Atmospheric Dispersion,Yes
9,fire_count_NW_lag_1,Fires_NW(t-1),Transboundary Biomass Plume,Yes


## 2. Feature Selection Ranking & Lasso Shrinkage
Evaluated strictly on the pre-monsoon training period to prevent target leakage.

In [3]:
df_sel = pd.read_csv(TABLES_DIR / "feature_selection_summary.csv", index_col=0)
df_sel.head(20)

,Mutual_Information,Lasso_Coefficient,VIF,Lasso_Retained
PM2.5,0.930258,14.905842,6.104098,Retained
PM10,0.749278,0.000000,NaN,Zeroed-Out
pm25_roll3_mean,0.669310,0.000000,NaN,Zeroed-Out
pm25_roll7_mean,0.665237,6.442465,NaN,Retained
pm25_lag_1,0.627401,0.000000,5.482748,Zeroed-Out
pm25_lag_2,0.550887,0.051810,NaN,Retained
pm25_lag_3,0.538474,0.000000,NaN,Zeroed-Out
pm25_lag_7,0.458639,0.000000,2.772130,Zeroed-Out
temp_range,0.373943,0.139553,NaN,Retained
humidity_mean,0.328283,-2.143837,2.937462,Retained


## 3. Class Imbalance Analysis for Severe Forecasting Target (Severe_Next_Day)

In [4]:
df_feat = pd.read_csv(PROCESSED_DATA_DIR / "master_features.csv")
print("Target severe_next_day value counts:")
print(df_feat["severe_next_day"].value_counts())
print(f"Severe episode prevalence: {df_feat['severe_next_day'].mean()*100:.2f}%")

Target severe_next_day value counts:
severe_next_day
0.0    3774
1.0     515
Name: count, dtype: int64
Severe episode prevalence: 12.01%
